<a href="https://colab.research.google.com/github/vijayrohara01-retro-dev/cv_cpp_hardware_optimization/blob/main/cv_cpp_hardware_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision: C++ Hardware Kernel Optimization
**Author:** Vijay Rohara | Independent Technical Consultant

### **Architecture Overview**
When deploying computer vision models on edge hardware (such as NVIDIA Jetson boards), pure Python implementations often introduce severe CPU to GPU latency. This project demonstrates how to bypass Python overhead by writing a custom C++ kernel to handle low level array manipulation (e.g. pixel intensity capping), compiling it as a shared library, and seamlessly integrating it back into the Python CV pipeline using 'ctypes'.

**Core Stack:** C++, Python, OpenCV, C-Types

In [ ]:
%%writefile edge_kernel.cpp
#include <iostream>

extern "C" {
    // Simulates a low-level C++ hardware optimization for image arrays
    void optimize_frame_buffer(int* frame_data, int size, int threshold) {
        for(int i = 0; i < size; i++) {
            if (frame_data[i] > threshold) {
                frame_data[i] = threshold; // Cap pixel intensity
            }
        }
    }
}

Writing edge_kernel.cpp


In [ ]:
!g++ -fPIC -shared -o libedge.so edge_kernel.cpp

### **Python Interoperability & OpenCV Processing**
Once the C++ shared library is compiled 'libedge.so', we use OpenCV to ingest a grayscale image array. The flattened array is passed directly to the C++ kernel in memory. This decoupled architecture allows us to maintain Python's ease of use for high level routing while leveraging C++ for computationally expensive, pixel by pixel operations.

In [ ]:
import cv2
import ctypes
import numpy as np
import urllib.request

# 1. Download a sample image
url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
urllib.request.urlretrieve(url, "sample.jpg")

# 2. Image Processing via OpenCV
image = cv2.imread("sample.jpg", cv2.IMREAD_GRAYSCALE)
flattened_image = image.flatten().astype(np.int32)

# 3. Load the C++ Shared Library
edge_lib = ctypes.CDLL('./libedge.so')
edge_lib.optimize_frame_buffer.argtypes = [np.ctypeslib.ndpointer(dtype=np.int32), ctypes.c_int, ctypes.c_int]

# 4. Pass the image array to C++ for hardware-level processing
print(f"Max pixel value before C++ processing: {flattened_image.max()}")
edge_lib.optimize_frame_buffer(flattened_image, len(flattened_image), 150)
print(f"Max pixel value after C++ processing: {flattened_image.max()}")

# Reshape back to image format
processed_image = flattened_image.reshape(image.shape)

Max pixel value before C++ processing: 248
Max pixel value after C++ processing: 150
